<a href="https://colab.research.google.com/github/widura26/recap-automation/blob/main/recap_automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## List BOM
- Elektrik + Pre Series ✅ ✅
- De-Scoping ✅
- Interior ✅ ✅
- TMS ✅
- CKD VVVF ✅
- CKD Pantograph ✅
- CKD SIV ✅
- Elektrik ✅ ✅
- Komponen Utama ✅ ✅
- Sistem Mekanik ✅ ✅
- Raw Material Interior ✅
- Bogie KIT ✅
- Raw Material Bogie ✅
- Realisasi Raw Material ✅
- Carbody
- Fastening Mekanik
- Fastening Bogie
- Fastening Interior
- Crashwrothiness
- Welding
- Welding Bogie
- Consumable Tools ✅
- Consumable Series
- Welding WS BWI
- Jig Tools 6TS Pelokalan
- Jig Tool
- Tools ✅
- Tools & Consumable Tools Perbaikan ✅ ✅
- Consumable Pre Series ✅✅

### kurang
- Carbody
- Fastening Mekanik
- Fastening Bogie
- Fastening Interior
- Crashwrothiness
- Welding
- Welding Bogie
- Consumable Series
- Welding WS BWI
- Jig Tools 6TS Pelokalan
- Jig Tool

In [5]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
import re

# Grabbing data

In [43]:
creds, _ = default()
client = gspread.authorize(creds)

# 2. Buka Spreadsheet
nama_file_sheets = "bom_dataset"
spreadsheet = client.open(nama_file_sheets)

# --- LIST LENGKAP ALL 29 TARGET SHEET ---
DAFTAR_SHEET_TARGET = [
    "Elektrik + Pre Series",
    "De-Scoping",
    "Interior",
    "TMS",
    "CKD VVVF",
    "CKD Pantograph",
    "CKD SIV",
    "Elektrik",
    "Komponen Utama",
    "Sistem Mekanik",
    "Raw Material Interior",
    "Bogie KIT",
    "Raw Material Bogie",
    "Realisasi Raw Material",
    "Carbody",
    "Fastening Mekanik",
    "Fastening Bogie",
    "Fastening Interior",
    "Crashwrothiness",
    "Welding",
    "Welding Bogie",
    "Consumable Tools",
    "Consumable Series",
    "Welding WS BWI",
    "Jig Tools 6TS Pelokalan",
    "jig Tool",
    "Tools",
    "Tools & Consumable Tools Perbaikan",
    "Consumable Pre Series",
]

semua_tab = spreadsheet.worksheets()
sheet_names_aktual = [sheet.title for sheet in semua_tab]

def cari_sheet_aktual(nama_target):
    target_clean = nama_target.strip().lower()
    for actual in sheet_names_aktual:
        if actual.strip().lower() == target_clean:
            return actual
    target_alphanumeric = re.sub(r'[^a-z0-9]', '', target_clean)
    for actual in sheet_names_aktual:
        actual_alphanumeric = re.sub(r'[^a-z0-9]', '', actual.strip().lower())
        if target_alphanumeric == actual_alphanumeric:
            return actual
    return None

target_sheets_matched = []
for nama_target in DAFTAR_SHEET_TARGET:
    matched = cari_sheet_aktual(nama_target)
    if matched and matched not in target_sheets_matched:
        target_sheets_matched.append(matched)

print(f"[INFO] Memetakan {len(target_sheets_matched)} dari {len(DAFTAR_SHEET_TARGET)} target sheet.\n")

KATA_KUNCI = {
    # 'kode': ['kode material', 'kode barang', 'item code', 'part number', 'kode', 'material code', 'no. part', 'part no', 'k o d e'],
    'kode': ['Kode Material'],
    # 'nama': ['material / komponen', 'material/komponen', 'nama barang', 'nama material', 'description', 'komponen', 'material name', 'item description', 'nama komponen', 'uraian', 'nama/spesifikasi', 'Deskripsi Material'],
    'nama': ['material / komponen', 'material/komponen', 'deskripsi material'],
    # 'spek': ['spesifikasi', 'specification', 'spek', 'ukuran', 'dimensi', 'spec', 'Spesifikasi'],
    'spek': ['spesifikasi', 'specification', 'Spesifikasi', 'Specification'],
    # 'qty' : ['qty / ts', 'qty/ts', 'quantity', 'qty', 'jumlah', 'total qty', 'ts', 'qty total', 'total', 'vol', 'volume', 'qty/set', 'qty / set', 'set', 'Qty/TS (Series)']
    'qty' : ['Qty/TS (Series)']
}

rekap_data = []

for target_name in target_sheets_matched:
    sheet = spreadsheet.worksheet(target_name)
    data_mentah = sheet.get_all_values()

    if len(data_mentah) < 3:
        print(f"Sheet '{target_name}' -> [SKIP] Baris kosong.")
        continue

    # 1. CARI HEADER HINGGA BARIS 50 (Penting untuk Jig Tool, Consumable, dsb.)
    header_idx = -1
    idx_kode = idx_nama = idx_spek = idx_qty = None

    for r_idx in range(min(50, len(data_mentah))):
        row_str = [str(c).strip().lower() for c in data_mentah[r_idx]]
        row_text_full = " ".join(row_str)

        if any(k in row_text_full for k in ['kode', 'part', 'material', 'komponen', 'description', 'uraian']):
            for c_idx, cell in enumerate(row_str):
                if idx_kode is None and any(k in cell for k in KATA_KUNCI['kode']):
                    idx_kode = c_idx
                if idx_nama is None and any(k in cell for k in KATA_KUNCI['nama']):
                    idx_nama = c_idx
                if idx_spek is None and any(k in cell for k in KATA_KUNCI['spek']):
                    idx_spek = c_idx
                if idx_qty is None and any(k in cell for k in KATA_KUNCI['qty']):
                    idx_qty = c_idx

            if idx_kode is not None or idx_nama is not None:
                header_idx = r_idx
                break

    # Fallback jika header tidak ketemu lewat kata kunci
    if header_idx == -1: header_idx = 7
    if idx_kode is None: idx_kode = 4   # Kolom E
    if idx_nama is None: idx_nama = 11  # Kolom L
    if idx_spek is None: idx_spek = 13  # Kolom N
    if idx_qty is None:  idx_qty = 30   # Kolom AE

    baris_terproses = 0

    for row in data_mentah[header_idx + 1:]:
        max_idx = max(idx_kode, idx_nama, idx_spek, idx_qty)
        while len(row) <= max_idx + 10:
            row.append("")

        kode_material = str(row[idx_kode]).strip()
        nama_material = row[idx_nama]
        spesifikasi = row[idx_spek] if idx_spek < len(row) else ""

        # 2. SMARTEST QTY SEARCH: Ambil dari idx_qty, jika tidak ada cari angka pertama setelah kolom Nama
        raw_qty = row[idx_qty]

        valid_qty = None
        str_qty = str(raw_qty).strip().replace(',', '.').upper()

        # Ekstrak angka dari Qty Utama
        if str_qty not in ["N/A", "-", "#N/A", "#VALUE!", ""]:
            match = re.search(r'-?\d+(\.\d+)?', str_qty)
            if match:
                valid_qty = float(match.group(0))

        # Jika Qty Utama gagal/kosong, pindai 5 kolom di kanan idx_nama & idx_qty
        if valid_qty is None:
            for check_idx in range(idx_nama + 1, len(row)):
                val = str(row[check_idx]).strip().replace(',', '.')
                match = re.search(r'^-?\d+(\.\d+)?$', val)
                if match:
                    valid_qty = float(match.group(0))
                    break

        kata_abaikan = ["kode material", "kode", "total", "subtotal", "header", "item code", "part number", "k o d e"]

        if kode_material and kode_material.lower() not in kata_abaikan and valid_qty is not None:
            rekap_data.append([
                target_name,
                kode_material,
                nama_material,
                spesifikasi,
                valid_qty
            ])
            baris_terproses += 1

    print(f"Sheet '{target_name}' (Header Baris {header_idx+1}) -> {baris_terproses} baris terambil.")

# --- OUTPUT TO SPREADSHEET ---
df_rekap = pd.DataFrame(rekap_data, columns=[
    "Nama Sheet Sumber", "Kode Material", "Material / Komponen", "Spesifikasi", "Qty / TS"
])

print("\n--- RINGKASAN HASIL PER SHEET ---")
print(df_rekap.groupby("Nama Sheet Sumber").size())

nama_tab_rekap = "REKAP_BOM_TOTAL (Berdasarkan Kode Material) 2"
try:
    sheet_rekap = spreadsheet.worksheet(nama_tab_rekap)
    sheet_rekap.clear()
except gspread.exceptions.WorksheetNotFound:
    sheet_rekap = spreadsheet.add_worksheet(title=nama_tab_rekap, rows="6000", cols="5")

data_ke_sheets = [df_rekap.columns.values.tolist()] + df_rekap.values.tolist()
sheet_rekap.update(data_ke_sheets)

print(f"\n[SUKSES] TOTAL {len(df_rekap)} baris data berhasil ditarik dari {df_rekap['Nama Sheet Sumber'].nunique()} sheet BOM.")

Streaming output truncated to the last 5000 lines.
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main valid qty
main

# Read data

In [38]:
urx = "https://docs.google.com/spreadsheets/d/1Ibki18gicAFziEx1urTrvDv_KkzrkMMnRrbwqNYpOkw/export?format=csv&gid=1382190492"

data = pd.read_csv(urx, low_memory=False)
data.head()

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
0,Elektrik + Pre Series,B336E12101,ELVP TC,Refer to dwg. 33.6-E12101 & 86.2-E12104 86.2-E...,2.0
1,Elektrik + Pre Series,B336E12201,ELVP M1,Refer to dwg. 33.6-E12201 & 86.2-E12204 86.2-E...,3.0
2,Elektrik + Pre Series,B336E12301,ELVP M2,Refer to dwg. 33.6-E12301 & 86.2-E12304 86.2-E...,3.0
3,Elektrik + Pre Series,B336E12401,ELVP T1,"Refer to dwg. 33.6-E12401 & 86.2-E12404, 86.2...",2.0
4,Elektrik + Pre Series,B336E12501,ELVP T2,Refer to dwg. 33.6-E12501 & 86.2-E12504 86.2-E...,1.0


# Analyze data

In [7]:
ckdpantographData = data[data["Nama Sheet Sumber"] == "CKD Pantograph"]
ckdpantographData[ckdpantographData["Material / Komponen"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
1862,CKD Pantograph,B52TP5088,NaN,NaN,1.0
1863,CKD Pantograph,B52TP5089,NaN,NaN,1.0
1864,CKD Pantograph,B52TP5090,NaN,NaN,1.0


In [8]:
ckdsivdata = data[data["Nama Sheet Sumber"] == "CKD SIV"]
ckdsivdata[ckdsivdata["Material / Komponen"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
1977,CKD SIV,B42CD0416,NaN,NaN,0.0
1982,CKD SIV,B44LA0008,NaN,NaN,100.0


In [9]:
carbodydata = data[data["Nama Sheet Sumber"] == "Carbody"]
carbodydata[carbodydata["Material / Komponen"].isna()].head()

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
3418,Carbody,BRACKET,NaN,NaN,0.0


In [10]:
fasteningmekanikdata = data[data["Nama Sheet Sumber"] == "Fastening Mekanik"]
fasteningmekanikdata[fasteningmekanikdata["Material / Komponen"].isna()].head()

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
4159,Fastening Mekanik,B46NG0424,NaN,NaN,432.0


In [11]:
rrmdata = data[data["Nama Sheet Sumber"] == "Realisasi Raw Material"]
rrmdata[rrmdata["Material / Komponen"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
3295,Realisasi Raw Material,A11BU0050,NaN,NaN,0.33
3316,Realisasi Raw Material,A11ND0160,NaN,NaN,0.04
3325,Realisasi Raw Material,A11ZJ0010,NaN,NaN,10.04
3346,Realisasi Raw Material,A15ND0230,NaN,NaN,0.05
3348,Realisasi Raw Material,A15SA0240,NaN,NaN,0.08
3350,Realisasi Raw Material,A15ZD0160,NaN,NaN,0.05
3374,Realisasi Raw Material,A11AA0045,NaN,NaN,3.00
3376,Realisasi Raw Material,A11AA0090,NaN,NaN,2.00


In [12]:
result = data[data["Material / Komponen"].isna()]
result.groupby("Nama Sheet Sumber").size()

,0
Nama Sheet Sumber,
CKD Pantograph,3
CKD SIV,2
Carbody,1
Fastening Mekanik,1
Realisasi Raw Material,8


In [13]:
spesifikasiisnaresult = data[data["Spesifikasi"].isna()]
spesifikasiisnaresult.groupby("Nama Sheet Sumber").size()

,0
Nama Sheet Sumber,
Bogie KIT,1
CKD Pantograph,3
CKD SIV,2
Carbody,1
Consumable Pre Series,2
Consumable Series,14
Consumable Tools,11
Crashwrothiness,1
De-Scoping,25


In [14]:
bkdata = data[data["Nama Sheet Sumber"] == "Bogie KIT"]
bkdata[bkdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
3138,Bogie KIT,B52TP3101,Eathing Brush,NaN,24.0


In [15]:
ckdpdata = data[data["Nama Sheet Sumber"] == "CKD Pantograph"]
ckdpdata[ckdpdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
1862,CKD Pantograph,B52TP5088,NaN,NaN,1.0
1863,CKD Pantograph,B52TP5089,NaN,NaN,1.0
1864,CKD Pantograph,B52TP5090,NaN,NaN,1.0


In [16]:
cdata = data[data["Nama Sheet Sumber"] == "Carbody"]
cdata[cdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
3418,Carbody,BRACKET,NaN,NaN,0.0


In [39]:
cpsdata = data[data["Nama Sheet Sumber"] == "Consumable Pre Series"]
cpsdata[cpsdata["Spesifikasi"].isna()]
#bermasalah nih kalkulasi quantitynya.

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
6989,Consumable Pre Series,D66UQ0001#BC,LAKBAN HITAM,NaN,35.0
7060,Consumable Pre Series,D68QH0125,ANTISOL E 125,NaN,3.0


In [18]:
csdata = data[data["Nama Sheet Sumber"] == "Consumable Series"]
csdata[csdata["Spesifikasi"].isna()] #bermasalah nih ada beberapa data yang nggak masuk

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
5505,Consumable Series,D66UQ0001#BC,LAKBAN HITAM,NaN,408.0
5575,Consumable Series,D68QH0125,ANTISOL E 125,NaN,60.0
5599,Consumable Series,D82WL0001,BENANG NILON,NaN,120.0
5619,Consumable Series,D68QH1004,LEM G,NaN,48.0
5630,Consumable Series,D66UQ0001#BC,LAKBAN HITAM,NaN,12.0
5637,Consumable Series,D31WH0506,TECTYL 506,NaN,12.0
5652,Consumable Series,D67UK0002,MAJUN,NaN,24.0
5666,Consumable Series,D68QG0005,Danagloss Polyester Putty ( Dempul ),NaN,6.0
5667,Consumable Series,D62QE0088#HB,Sigmafast 278 + Hardener,NaN,10.0
5668,Consumable Series,D63SA0017,Kertas Gosok No.1,NaN,20.0


In [19]:
crsdata = data[data["Nama Sheet Sumber"] == "Crashwrothiness"]
crsdata[crsdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
5134,Crashwrothiness,B43KR1001,NUT_HEX_M10XP1.5_10T,NaN,15.0


In [20]:
dsdata = data[data["Nama Sheet Sumber"] == "De-Scoping"]
dsdata[dsdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
363,De-Scoping,D66UQ0001#BC,LAKBAN HITAM,NaN,408.0
433,De-Scoping,D68QH0125,ANTISOL E 125,NaN,60.0
457,De-Scoping,D82WL0001,BENANG NILON,NaN,120.0
477,De-Scoping,D68QH1004,LEM G,NaN,48.0
488,De-Scoping,D66UQ0001#BC,LAKBAN HITAM,NaN,12.0
495,De-Scoping,D31WH0506,TECTYL 506,NaN,12.0
510,De-Scoping,D67UK0002,MAJUN,NaN,24.0
524,De-Scoping,D68QG0005,Danagloss Polyester Putty ( Dempul ),NaN,6.0
525,De-Scoping,D62QE0088#HB,Sigmafast 278 + Hardener,NaN,10.0
526,De-Scoping,D63SA0017,Kertas Gosok No.1,NaN,20.0


In [21]:

elektrikData = data[data["Nama Sheet Sumber"] == "Elektrik"]
elektrikData[elektrikData["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
2198,Elektrik,D83SP0101,BALLPOINT OPF,NaN,10.0
2281,Elektrik,B52TG1213CPU,CPU,NaN,2.0
2282,Elektrik,B52TG1213AG,GPS ANTENA,NaN,2.0
2283,Elektrik,B52TG1213HMI,PIS & PAS HMI,NaN,2.0
2284,Elektrik,B52TG1213SW,ETH SWITCH,NaN,12.0
2285,Elektrik,B52TG1213AM,AMPLIFIER,NaN,12.0
2286,Elektrik,B52TG1213FDI,FRONT DESTINATION INDICATOR,NaN,2.0
2287,Elektrik,B52TG1213RTO,SIDE LED RUNNING TEXT,NaN,24.0
2288,Elektrik,B52TG1213WLCD,PAS MONITOR,NaN,96.0
2289,Elektrik,B52TG1213DIG,IO PIS & PAS MODULE,NaN,2.0


In [22]:
elektrikData = data[data["Nama Sheet Sumber"] == "Elektrik + Pre Series"]
elektrikData[elektrikData["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
161,Elektrik + Pre Series,A04WG3032,Hose Vinil 30X32 MM,NaN,48.0
207,Elektrik + Pre Series,D83SP0101,BALLPOINT OPF,NaN,10.0
295,Elektrik + Pre Series,B43KC0407,NUT_HEX_M4xP0.7_SUS,NaN,16.0
300,Elektrik + Pre Series,B52TG1213CPU,CPU,NaN,2.0
301,Elektrik + Pre Series,B52TG1213AG,GPS ANTENA,NaN,2.0
302,Elektrik + Pre Series,B52TG1213HMI,PIS & PAS HMI,NaN,2.0
303,Elektrik + Pre Series,B52TG1213SW,ETH SWITCH,NaN,12.0
304,Elektrik + Pre Series,B52TG1213AM,AMPLIFIER,NaN,12.0
305,Elektrik + Pre Series,B52TG1213FDI,FRONT DESTINATION INDICATOR,NaN,2.0
306,Elektrik + Pre Series,B52TG1213RTO,SIDE LED RUNNING TEXT,NaN,24.0


In [24]:
fmdata = data[data["Nama Sheet Sumber"] == "Fastening Mekanik"]
fmdata[fmdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
3952,Fastening Mekanik,B42CA101530,BOLT_HEX_JIS B1180,NaN,22.0
4159,Fastening Mekanik,B46NG0424,NaN,NaN,432.0
4435,Fastening Mekanik,B45MS0225,PIN_SPLIT_DIA2.5X25_SUS,NaN,4.0
4474,Fastening Mekanik,B41BE040710,SCREW_CSK_HEAD_JIS B1111,NaN,24.0
4534,Fastening Mekanik,B44LF0005,WASHER_SPRING_M5_SUS,NaN,96.0


In [26]:
jtdata = data[data["Nama Sheet Sumber"] == "Jig Tool"]
jtdata[jtdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
6271,Jig Tool,B42CH2015,DYNA BOLT M20 X 150 MM,NaN,50.0


In [29]:
jt6data = data[data["Nama Sheet Sumber"] == "Jig Tools 6TS Pelokalan"]
jt6data[jt6data["Spesifikasi"].isna()] #Bermasalah ini di Qty / TS

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
5714,Jig Tools 6TS Pelokalan,C86VC0001,1 SET OBENG KECIL -/+,NaN,10.0
5717,Jig Tools 6TS Pelokalan,C86UH6242,"Cleaner, TIP Solder Iron Mesh",NaN,4.0
5778,Jig Tools 6TS Pelokalan,D83SK0100,RANTAI PLASTIK WARNA MERAH,NaN,50.0
5928,Jig Tools 6TS Pelokalan,D66QG4608,LEM AICA,NaN,5.0
5932,Jig Tools 6TS Pelokalan,D83SP0101,BALLPOINT OPF,NaN,50.0
5953,Jig Tools 6TS Pelokalan,D29TC0010,SOLDER RH.50 (TIMAH),NaN,5.0
5957,Jig Tools 6TS Pelokalan,D66UG0015,ISOLASI KERTAS,NaN,100.0
5958,Jig Tools 6TS Pelokalan,D83SK0050,POLICE LINE MOTIF HITAM KUNING,NaN,10.0
5962,Jig Tools 6TS Pelokalan,D83SP02001,SAPU LANTAI,NaN,10.0
5963,Jig Tools 6TS Pelokalan,D83SP02002,CIKRAK,NaN,10.0


In [30]:
kudata = data[data["Nama Sheet Sumber"] == "Komponen Utama"]
kudata[kudata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
2425,Komponen Utama,B40KA1201,Underframe Assembly,NaN,12.0
2426,Komponen Utama,B40KA1201,Side Wall Assembly,NaN,24.0
2427,Komponen Utama,B40KA1201,Roof Assembly of Carbody,NaN,12.0
2428,Komponen Utama,B40KA1201,End Wall Assembly of Carbody,NaN,22.0
2429,Komponen Utama,B40KA1201,Rain Gutter,NaN,12.0
2430,Komponen Utama,B40KA1201,Partition Frame,NaN,2.0
2431,Komponen Utama,B40KA1201,Cab Mask,NaN,2.0
2432,Komponen Utama,B40KA1201,Floor,NaN,12.0
2472,Komponen Utama,B463A170021,First Aid,NaN,12.0
2482,Komponen Utama,B52AC0143,Driver cab's door,NaN,4.0


In [31]:
rawdata = data[data["Nama Sheet Sumber"] == "Realisasi Raw Material"]
rawdata[rawdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
3295,Realisasi Raw Material,A11BU0050,NaN,NaN,0.33
3316,Realisasi Raw Material,A11ND0160,NaN,NaN,0.04
3325,Realisasi Raw Material,A11ZJ0010,NaN,NaN,10.04
3346,Realisasi Raw Material,A15ND0230,NaN,NaN,0.05
3348,Realisasi Raw Material,A15SA0240,NaN,NaN,0.08
3350,Realisasi Raw Material,A15ZD0160,NaN,NaN,0.05
3374,Realisasi Raw Material,A11AA0045,NaN,NaN,3.00
3376,Realisasi Raw Material,A11AA0090,NaN,NaN,2.00


In [33]:
smdata = data[data["Nama Sheet Sumber"] == "Sistem Mekanik"]
smdata[smdata["Spesifikasi"].isna()] #Anomali nih Qty nya

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
2575,Sistem Mekanik,B48BQ8518,SPRING,NaN,6.00
2600,Sistem Mekanik,A11ND0020,STEEL PLATE,NaN,0.01
2602,Sistem Mekanik,A15ND0030,ROUND BAR,NaN,0.10
2624,Sistem Mekanik,B51QN6161,ROUND PLUG,NaN,1.00
2729,Sistem Mekanik,A16HD02178,STEEL PIPE,NaN,2.10
2849,Sistem Mekanik,A11RF0020,STEEL PLATE,NaN,1.50
2852,Sistem Mekanik,A11RF0020,STEEL PLATE,NaN,1.50
2975,Sistem Mekanik,A11AB0023,STEEL PLATE,NaN,0.80
3026,Sistem Mekanik,B51ND1025B,BALLVALVE,NaN,2.00


In [34]:
toolsData = data[data["Nama Sheet Sumber"] == "Tools"]
toolsData[toolsData["Spesifikasi"].isna()] #Qty nya juga salah ini, nggak akurat

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
6410,Tools,C86LE0004,"PALU 0,5 LBs",NaN,0.0
6531,Tools,D88HH00301,ISI CUTTER BESAR,NaN,0.0
6551,Tools,C86ME1109,TANG JEPIT,NaN,0.0
6556,Tools,C88QE0250,KIKIR PLATE + GAGANG,NaN,1.0
6571,Tools,D79SP0020,KAPI SCRUB,NaN,0.0
6573,Tools,C79RA0003,LEPAN,NaN,0.0
6574,Tools,C79RA0004,CETOK,NaN,0.0
6575,Tools,C90HI0001,TIMBANGAN DUDUK,NaN,0.0
6855,Tools,C86VC0004,"OBENG (-) 5""",NaN,2.0
6856,Tools,C87SE0008,WIRE STRIPPER (PENGUPAS),NaN,1.0


In [35]:
tctpdata = data[data["Nama Sheet Sumber"] == "Tools & Consumable Tools Perbaikan"]
tctpdata[tctpdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
6870,Tools & Consumable Tools Perbaikan,D65SW0601,"GAS NITROGEN GAS (UHP) 99,995% @7M3",NaN,36.0
6875,Tools & Consumable Tools Perbaikan,D63SA1081,FLAP DISC GRIT Z60 100 X 16 MM,NaN,3.0
6893,Tools & Consumable Tools Perbaikan,D69RL008024,FILLER METAL GTAW 2.4 mm ER 80S-G,NaN,1.0
6897,Tools & Consumable Tools Perbaikan,D65SW0400,"GAS ARGON MURNI 99,9% @7M3",NaN,12.0
6924,Tools & Consumable Tools Perbaikan,D64QF0017,THINNER,NaN,3.0
6926,Tools & Consumable Tools Perbaikan,C86UM0320,STEKER,NaN,2.0
6939,Tools & Consumable Tools Perbaikan,D84TP1003,WELDING HELMET KEDOK LAS,NaN,2.0
6942,Tools & Consumable Tools Perbaikan,D83SP1005,SNOWMAN WHITE MARKER,NaN,4.0
6943,Tools & Consumable Tools Perbaikan,D83SP1006,PERMANENT MARKER HITAM,NaN,4.0
6945,Tools & Consumable Tools Perbaikan,C86LE0004,"PALU 0,5 LBs",NaN,2.0


- Consumable Pre Series (Masalah di quantity)
- Consumable Series (Ada data yang nggak masuk)
- Jig Tools 6TS Pelokalan (Bermasalah di QTY / TS)
- Sistem Mekanik (Bermasalah di QTY / TS)
- Tools (bermasalah di qty / ts)


# New Section

In [ ]:
bomdata = data.groupby("Nama Sheet Sumber").size()
bomdata = bomdata.to_frame(name="Jumlah")
bomdata

,Jumlah
Nama Sheet Sumber,
Bogie KIT,44
CKD Pantograph,266
CKD SIV,119
CKD VVVF,147
Carbody,246
Consumable Pre Series,93
Consumable Series,178
Consumable Tools,275
Crashwrothiness,19


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
data[data['Material / Komponen'].isna() & (data['Qty / TS'] == 0)]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
1235,CKD SIV,B42CD0416,NaN,NaN,0.0
2677,Carbody,BRACKET,NaN,NaN,0.0


In [ ]:
bogieKitData = data[(data['Nama Sheet Sumber'] == 'Bogie KIT')]
bogieKitData

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
2374,Bogie KIT,B48BR0100,Driven Wheelset (MB),Sesuai dengan spesifikasi teknis 108/SPT/H100...,0.0
2375,Bogie KIT,B48BL0106,Gear Unit / Gear Box,Sesuai dengan spesifikasi teknis 106/SPT/H100...,0.0
2376,Bogie KIT,B48BC0119,Axle Box Housing,"02.0-M20002, 119/SPT/H1005MB051/22",0.0
2377,Bogie KIT,B48LA5959,Journal roller bearing,Sesuai dengan spesifikasi teknis 130/SPT/H1005...,0.0
2378,Bogie KIT,B48AH2004,Rubber Bush for Axle Box,Sesuai dengan spesifikasi teknis 119/SPT/H1005...,0.0
2379,Bogie KIT,B48BW1622,Axle Box Cap (for Grounding Device),02.0-M20006,0.0
2380,Bogie KIT,B48RD0015,Pole Wheel,02.0-E11014,0.0
2381,Bogie KIT,B48BW1122,Cap Cover,02.0-P23311,0.0
2382,Bogie KIT,B48BQ1122,Coil Spring,Sesuai dengan spesifikasi teknis 129/SPT/H10...,0.0
2383,Bogie KIT,B48DJ1122,Axle Spring Guide (2),Sesuai drawing B48DJ1122,0.0
